<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/notebooks/07_Job_Matching_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CareerLens AI - Final AI Pipeline
This notebook represents the complete end-to-end inference pipeline for the CareerLens AI product.
It implements: Resume Understanding, Job Matching, Gap Analysis, and RAG-based Recommendations.

## SECTION 1: Environment and Configuration

In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu spacy datasets pandas numpy
!python -m spacy download en_core_web_sm

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import spacy
import faiss
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

# Configurations
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RAG_LLM_NAME = "google/flan-t5-small" # Lightweight for Colab CPU/T4 compatibility
CLASSIFIER_MODEL_PATH = "./final_resume_classifier" # Fallback to huggingface if not local

print("Environment configured.")

## SECTION 2: Load Existing Resume Pipeline

In [ ]:
# Load Embedding Model
print("Loading Sentence Transformer...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Load Fine-Tuned Classifier from Phase 6 (If exists, else load a fallback zero-shot)
print("Loading Classifier...")
try:
    classifier_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_PATH)
    classifier_model = AutoModelForSequenceClassification.from_pretrained(CLASSIFIER_MODEL_PATH)
    resume_classifier = pipeline("text-classification", model=classifier_model, tokenizer=classifier_tokenizer)
except:
    print("Local classifier not found. Using zero-shot fallback for demo purposes.")
    resume_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Load NLP for Entity Extraction
nlp = spacy.load("en_core_web_sm")

## SECTION 3: Resume Structured Profile

In [ ]:
def extract_resume_profile(resume_text):
    """Extracts structured candidate profile from raw resume text."""
    # 1. Classification
    try:
        if resume_classifier.task == "text-classification":
            cls_result = resume_classifier(resume_text[:512], truncation=True)[0]
            category = cls_result['label']
            confidence = cls_result['score']
        else:
            categories = ["HR", "Information-Technology", "Sales", "Finance", "Engineering", "Healthcare"]
            cls_result = resume_classifier(resume_text[:512], candidate_labels=categories)
            category = cls_result['labels'][0]
            confidence = cls_result['scores'][0]
    except:
        category = "UNKNOWN"
        confidence = 0.0

    # 2. Entity Extraction (Simulated zero-shot/regex for missing 04.2 notebook)
    doc = nlp(resume_text)
    # Highly simplified heuristic extraction for demo
    skills = [ent.text for ent in doc.ents if ent.label_ in ["ORG", "PRODUCT"]][:10]
    skills = list(set([s.lower() for s in skills if len(s) > 2])) # Normalize
    
    # For strict anti-hallucination, if we don't confidently find it, mark UNKNOWN or empty
    profile = {
        "skills": skills if skills else ["UNKNOWN"],
        "education": ["UNKNOWN"], # Would require layout/context analysis
        "experience": ["UNKNOWN"],
        "projects": ["UNKNOWN"],
        "certifications": ["UNKNOWN"],
        "languages": ["UNKNOWN"],
        "summary": resume_text[:200] + "...",
        "predicted_category": category,
        "category_confidence": float(confidence)
    }
    return profile

# Test extraction
sample_resume = "Experienced Python Developer with 3 years at Microsoft. Skilled in PyTorch, SQL, and Docker. B.Sc in Computer Science."
candidate_profile = extract_resume_profile(sample_resume)
print(json.dumps(candidate_profile, indent=2))

## SECTION 4: Job Data Collection

In [ ]:
class JobSource:
    def get_jobs(self):
        raise NotImplementedError

class DemoJobSource(JobSource):
    def get_jobs(self):
        # Demo jobs clearly labeled
        return [
            {
                "job_id": "DEMO-001",
                "title": "Machine Learning Engineer",
                "company": "AI Startup DEMO",
                "location": "Remote",
                "description": "We are looking for an ML Engineer. Required skills: Python, PyTorch, SQL, AWS, Docker.",
                "url": "https://demo.com/jobs/1",
                "source": "Demo Data",
                "date": "2026-01-01"
            },
            {
                "job_id": "DEMO-002",
                "title": "Backend Developer",
                "company": "Tech Corp DEMO",
                "location": "New York",
                "description": "Looking for a developer with Python, Django, SQL, and Kubernetes experience.",
                "url": "https://demo.com/jobs/2",
                "source": "Demo Data",
                "date": "2026-01-02"
            }
        ]

job_source = DemoJobSource()
jobs_db = job_source.get_jobs()
print(f"Loaded {len(jobs_db)} jobs from DemoJobSource.")

## SECTION 5: Job Information Extraction

In [ ]:
def extract_job_info(job_desc):
    """Extracts requirements from job description (Mocked for NLP pipeline)."""
    # In production, use Zero-Shot NER or LLM.
    doc = nlp(job_desc)
    extracted_skills = [ent.text.lower() for ent in doc.ents if ent.label_ in ["ORG", "PRODUCT"]]
    
    # Fallback keyword matching for demo stability
    keywords = ["python", "pytorch", "sql", "aws", "docker", "kubernetes", "django"]
    for kw in keywords:
        if kw in job_desc.lower() and kw not in extracted_skills:
            extracted_skills.append(kw)

    return {
        "required_skills": list(set(extracted_skills)),
        "preferred_skills": [],
        "experience_requirements": [],
        "education_requirements": []
    }

for job in jobs_db:
    job["extracted"] = extract_job_info(job["description"])
    print(f"Job: {job['title']} -> Requirements: {job['extracted']['required_skills']}")

## SECTION 6: Resume and Job Embeddings

In [ ]:
# Generate embeddings for semantic similarity matching
def generate_embedding(text):
    return embedding_model.encode(text, convert_to_tensor=True)

resume_embedding = generate_embedding(sample_resume)

for job in jobs_db:
    job["embedding"] = generate_embedding(job["description"])

## SECTION 7: Job Matching Engine & SECTION 8: Skill Gap Analysis

In [ ]:
def calculate_match(candidate_profile, resume_embedding, job):
    """Calculates an explainable match score combining semantics and exact skill overlap."""
    # 1. Semantic Similarity (0 to 1)
    semantic_score = util.cos_sim(resume_embedding, job["embedding"])[0][0].item()
    
    # 2. Skill Gap Analysis
    req_skills = set(job["extracted"]["required_skills"])
    cand_skills = set(candidate_profile["skills"])
    # Fallback to literal text search for anti-hallucination evidence
    matched_skills = [s for s in req_skills if s in cand_skills or s in sample_resume.lower()]
    missing_skills = [s for s in req_skills if s not in matched_skills]
    
    skill_score = len(matched_skills) / len(req_skills) if req_skills else 1.0
    
    # Final Score Weighting: 60% Skills, 40% Semantic
    final_score = (skill_score * 0.60) + (semantic_score * 0.40)
    
    evidence = []
    for skill in matched_skills:
        evidence.append({"skill": skill, "status": "FOUND", "source": "Resume Text"})
    for skill in missing_skills:
        evidence.append({"skill": skill, "status": "NOT_FOUND", "source": "Job Description Requirements"})

    return {
        "job_id": job["job_id"],
        "title": job["title"],
        "company": job["company"],
        "score": round(final_score * 100, 1),
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "evidence": evidence
    }

matches = [calculate_match(candidate_profile, resume_embedding, job) for job in jobs_db]
matches = sorted(matches, key=lambda x: x["score"], reverse=True)

print("--- JOB RANKING ---")
for m in matches:
    print(f"{m['title']} - Score: {m['score']}% (Missing: {m['missing_skills']})")

## SECTION 9: RAG Knowledge Base & SECTION 10: Grounded Career Recommendations

In [ ]:
# RAG Setup: Load LLM
generator = pipeline("text2text-generation", model=RAG_LLM_NAME)

# Knowledge Base (Mock curated learning materials)
knowledge_base = [
    "AWS documentation provides tutorials on EC2, S3, and cloud architecture deployment.",
    "Docker is essential for containerizing applications. Learn Docker via Docker Hub official guides.",
    "Kubernetes orchestrates containers. It is required for scaling backend services.",
    "PyTorch is a machine learning framework used for deep learning and neural networks."
]
kb_embeddings = embedding_model.encode(knowledge_base)
dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings)

def generate_grounded_recommendation(missing_skill):
    """Uses RAG to find resources and generate anti-hallucination recommendation."""
    query_emb = embedding_model.encode([f"How to learn {missing_skill}?"], convert_to_tensor=False)
    D, I = index.search(query_emb, k=1)
    retrieved_text = knowledge_base[I[0][0]]
    
    # Strict anti-hallucination prompt
    prompt = f"Context: {retrieved_text}\nCandidate is missing {missing_skill}. Based ONLY on the context, what should they learn?"
    res = generator(prompt, max_new_tokens=50)[0]['generated_text']
    return {
        "skill": missing_skill,
        "recommendation": res,
        "rag_source": retrieved_text
    }

print("\n--- CAREER ROADMAP (RAG) ---")
best_job = matches[0]
for gap in best_job["missing_skills"]:
    rec = generate_grounded_recommendation(gap)
    print(f"Gap: {gap}\nRecommendation: {rec['recommendation']}\nSource: {rec['rag_source']}\n")

## SECTION 11: End-to-End Evaluation

In [ ]:
print("Evaluation Notes:")
print("- Classification Model: Evaluated in 06_FineTuning.ipynb (Macro F1 used).")
print("- Job Matching: Evaluated qualitatively. Requires a labeled manually ranked evaluation set (NDCG or Precision@K) before true production tuning.")
print("- RAG: Hallucination rate is minimized by providing strict 'Based ONLY on context' prompt.")

## SECTION 12: End-to-End Inference Demo

In [ ]:
def inference_demo(cv_text):
    print("========================================")
    print("          CAREERLENS AI DEMO            ")
    print("========================================\n")
    
    print("1. RESUME UNDERSTANDING")
    profile = extract_resume_profile(cv_text)
    print(f"Predicted Category: {profile['predicted_category']} (Conf: {profile['category_confidence']:.2f})")
    print(f"Extracted Skills: {profile['skills']}\n")
    
    print("2. FIND JOBS FOR ME")
    cv_emb = generate_embedding(cv_text)
    results = [calculate_match(profile, cv_emb, j) for j in jobs_db]
    results = sorted(results, key=lambda x: x["score"], reverse=True)
    
    top_job = results[0]
    print(f"Top Match: {top_job['title']} at {top_job['company']} ({top_job['score']}%)")
    print(f"Matched Skills (FOUND): {top_job['matched_skills']}")
    print(f"Missing Skills (NOT_FOUND): {top_job['missing_skills']}\n")
    
    print("3. RAG CAREER RECOMMENDATIONS")
    for gap in top_job["missing_skills"]:
        rec = generate_grounded_recommendation(gap)
        print(f"- To learn {gap}: {rec['recommendation']}")

# Run Demo
inference_demo(sample_resume)